# Stone-Level Detection Evaluation (Multi-Model)

This notebook evaluates segmentation performance at the **individual stone level** rather than pixel level, comparing **all three model configurations** (3-channel, 4-channel, 7-channel) in a single run.

**Key insight**: Individual stones in the ground truth are separated by black (background) pixels. Using connected component analysis, we can identify each stone as a distinct object and evaluate:
- **Detection Rate**: How many GT stones were detected (regardless of class)?
- **Classification Accuracy**: Of detected stones, how many have the correct class?
- **Per-Class Detection**: Detection rates broken down by masonry type
- **Merge Detection**: How many GT stones were incorrectly merged into single predictions?

**Important**: This notebook uses **strict 1:1 matching** — each predicted stone region can only be assigned to ONE ground truth stone. If a model merges multiple GT stones into one prediction, only the best-matching GT stone counts as detected; the others are marked as 'merged' (undetected).

## Workflow:
1. **Set parameters** (paths for GT and all 3 predictions, detection threshold)
2. **Extract individual stones** from GT using connected components
3. **Extract predicted regions** from each prediction
4. **Match stones** with 1:1 constraint and detect merges
5. **Compare results** across models

---
## Cell 1: Parameters

Set your paths and detection thresholds here. Re-run subsequent cells after changing these.

In [ ]:
# ============================================================
# PARAMETERS - EDIT THESE
# ============================================================

# Ground truth mask path
GT_MASK_PATH = "/path/to/ground_truth_mask.png"

# Prediction paths for all three models
PRED_PATHS = {
    '3-channel': "/path/to/3channel_prediction.png",   # Geometry-only (normal maps)
    '4-channel': "/path/to/4channel_prediction.png",   # Appearance-only (RGB + alpha)
    '7-channel': "/path/to/7channel_prediction.png"    # Combined (RGB + alpha + normals)
}

# Output directory for results
OUTPUT_DIR = "/path/to/output/"

# Stone Detection Parameters
# ---------------------------
# IoU threshold for considering a stone "detected"
# 0.5 = standard object detection threshold (COCO-style)
# 0.7 = stricter threshold requiring better boundary alignment
# 0.9 = very strict, only near-perfect matches
IOU_THRESHOLD = 0.5

# Minimum stone size (in pixels) to consider
# Helps filter out tiny noise artifacts
MIN_STONE_SIZE = 100  # pixels

# Class names (should match your color scheme)
CLASS_NAMES = ['Background', 'Ashlar', 'Polygonal', 'Quarry Stone']

# Model display colors for charts
MODEL_COLORS = {
    '3-channel': '#e74c3c',  # Red
    '4-channel': '#3498db',  # Blue
    '7-channel': '#2ecc71'   # Green
}

# Class colors for visualizations
CLASS_COLORS_DICT = {
    'Background': '#000000',
    'Ashlar': '#0000FF',
    'Polygonal': '#FF0000',
    'Quarry Stone': '#FFFF00'
}

print(f"Ground Truth: {GT_MASK_PATH}")
print(f"\nPrediction Paths:")
for model_name, path in PRED_PATHS.items():
    print(f"  {model_name}: {path}")
print(f"\nOutput Dir: {OUTPUT_DIR}")
print(f"\nDetection Parameters:")
print(f"  IoU Threshold: {IOU_THRESHOLD}")
print(f"  Min Stone Size: {MIN_STONE_SIZE} pixels")

---
## Cell 2: Imports and Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import seaborn as sns
from PIL import Image
import pandas as pd
from scipy import ndimage
from scipy.optimize import linear_sum_assignment
from skimage.measure import label, regionprops
from typing import Dict, Tuple, List, Optional, Set
from dataclasses import dataclass, field
from collections import defaultdict
import os
import warnings
warnings.filterwarnings('ignore')

Image.MAX_IMAGE_PIXELS = None

# RGB to class mapping
RGB_TO_CLASS = {
    (0, 0, 0): 0,       # Black -> Background
    (0, 0, 255): 1,     # Blue -> Ashlar
    (255, 0, 0): 2,     # Red -> Polygonal
    (255, 255, 0): 3    # Yellow -> Quarry Stone
}

CLASS_COLORS = ['#000000', '#0000FF', '#FF0000', '#FFFF00']

print("Imports complete.")

---
## Cell 3: Data Classes and Helper Functions

In [ ]:
@dataclass
class Stone:
    """Represents a single stone (GT or predicted) with its properties."""
    stone_id: int
    class_id: int
    class_name: str
    pixel_count: int
    mask: np.ndarray  # Boolean mask for this stone's pixels
    centroid: Tuple[float, float]
    bbox: Tuple[int, int, int, int]  # (min_row, min_col, max_row, max_col)


@dataclass
class StoneMatch:
    """Result of matching a GT stone to prediction."""
    gt_stone: Stone
    status: str  # 'detected', 'merged', 'undetected'
    iou: float
    predicted_class: Optional[int]
    predicted_stone_id: Optional[int]  # ID of matched predicted stone
    is_correctly_classified: bool
    # For merged stones: which other GT stones share the same prediction
    merged_with_gt_ids: List[int] = field(default_factory=list)


def rgb_to_class_mask(rgb_image: np.ndarray, verbose: bool = True) -> np.ndarray:
    """
    Convert RGB mask to class indices.
    Handles unmapped colors by assigning to nearest class.
    """
    height, width = rgb_image.shape[:2]
    class_mask = np.zeros((height, width), dtype=np.uint8)
    
    for rgb_tuple, class_idx in RGB_TO_CLASS.items():
        color_mask = np.all(rgb_image == rgb_tuple, axis=2)
        class_mask[color_mask] = class_idx
    
    # Handle unmapped pixels (compression artifacts, anti-aliasing)
    mapped_pixels = np.zeros((height, width), dtype=bool)
    for rgb_tuple in RGB_TO_CLASS.keys():
        mapped_pixels |= np.all(rgb_image == rgb_tuple, axis=2)
    
    unmapped_count = np.sum(~mapped_pixels)
    if unmapped_count > 0 and verbose:
        print(f"    Note: {unmapped_count} pixels with unmapped colors -> mapped to nearest class")
        unmapped_indices = np.where(~mapped_pixels)
        for i in range(len(unmapped_indices[0])):
            y, x = unmapped_indices[0][i], unmapped_indices[1][i]
            pixel_rgb = rgb_image[y, x]
            min_dist = float('inf')
            nearest_class = 0
            for rgb_tuple, class_idx in RGB_TO_CLASS.items():
                dist = np.sqrt(np.sum((pixel_rgb.astype(float) - np.array(rgb_tuple).astype(float))**2))
                if dist < min_dist:
                    min_dist = dist
                    nearest_class = class_idx
            class_mask[y, x] = nearest_class
    
    return class_mask


def extract_stones_from_mask(
    class_mask: np.ndarray, 
    class_names: List[str],
    min_size: int = 100
) -> List[Stone]:
    """
    Extract individual stones from a class mask using connected component analysis.
    
    Each connected component of non-background pixels is considered one stone.
    The class of the stone is determined by majority vote of its pixels.
    """
    # Create binary mask of all stone pixels (non-background)
    stone_binary = (class_mask > 0).astype(np.uint8)
    
    # Label connected components
    labeled_array, num_features = ndimage.label(stone_binary)
    
    stones = []
    for region in regionprops(labeled_array):
        # Filter by size
        if region.area < min_size:
            continue
        
        # Create boolean mask for this stone
        stone_mask = (labeled_array == region.label)
        
        # Determine class by majority vote
        stone_classes = class_mask[stone_mask]
        class_id = int(np.bincount(stone_classes).argmax())
        
        stone = Stone(
            stone_id=region.label,
            class_id=class_id,
            class_name=class_names[class_id],
            pixel_count=region.area,
            mask=stone_mask,
            centroid=(region.centroid[0], region.centroid[1]),
            bbox=(region.bbox[0], region.bbox[1], region.bbox[2], region.bbox[3])
        )
        stones.append(stone)
    
    return stones


def calculate_iou(mask1: np.ndarray, mask2: np.ndarray) -> float:
    """Calculate IoU between two boolean masks."""
    intersection = np.logical_and(mask1, mask2).sum()
    union = np.logical_or(mask1, mask2).sum()
    if union == 0:
        return 0.0
    return intersection / union


print("Data classes and helper functions defined.")

---
## Cell 4: Matching Functions with 1:1 Constraint

In [ ]:
def compute_iou_matrix(
    gt_stones: List[Stone],
    pred_stones: List[Stone]
) -> np.ndarray:
    """
    Compute IoU matrix between all GT and predicted stones.
    
    Returns:
        Matrix of shape (n_gt, n_pred) with IoU values
    """
    n_gt = len(gt_stones)
    n_pred = len(pred_stones)
    
    iou_matrix = np.zeros((n_gt, n_pred))
    
    for i, gt_stone in enumerate(gt_stones):
        for j, pred_stone in enumerate(pred_stones):
            iou_matrix[i, j] = calculate_iou(gt_stone.mask, pred_stone.mask)
    
    return iou_matrix


def find_potential_merges(
    iou_matrix: np.ndarray,
    gt_stones: List[Stone],
    iou_threshold: float
) -> Dict[int, List[int]]:
    """
    Find which predicted stones could match multiple GT stones (potential merges).
    
    Returns:
        Dict mapping pred_stone_index -> list of GT stone indices that match it
    """
    pred_to_gt_candidates = defaultdict(list)
    
    n_gt, n_pred = iou_matrix.shape
    
    for gt_idx in range(n_gt):
        for pred_idx in range(n_pred):
            if iou_matrix[gt_idx, pred_idx] > 0:  # Any overlap
                pred_to_gt_candidates[pred_idx].append(gt_idx)
    
    # Filter to only those with multiple GT candidates
    merges = {k: v for k, v in pred_to_gt_candidates.items() if len(v) > 1}
    
    return merges


def match_stones_one_to_one(
    gt_stones: List[Stone],
    pred_stones: List[Stone],
    pred_mask: np.ndarray,
    iou_threshold: float = 0.5
) -> Tuple[List[StoneMatch], Dict]:
    """
    Match GT stones to predicted stones with strict 1:1 constraint.
    
    Uses Hungarian algorithm to find optimal assignment that maximizes total IoU.
    Each predicted stone can only be assigned to ONE GT stone.
    
    Returns:
        Tuple of (list of StoneMatch objects, merge statistics dict)
    """
    if len(pred_stones) == 0:
        # No predictions - all GT stones are undetected
        matches = [
            StoneMatch(
                gt_stone=gt_stone,
                status='undetected',
                iou=0.0,
                predicted_class=None,
                predicted_stone_id=None,
                is_correctly_classified=False,
                merged_with_gt_ids=[]
            )
            for gt_stone in gt_stones
        ]
        merge_stats = {'total_merges': 0, 'merged_gt_stones': 0, 'merge_groups': []}
        return matches, merge_stats
    
    # Compute IoU matrix
    iou_matrix = compute_iou_matrix(gt_stones, pred_stones)
    
    # Find potential merges BEFORE 1:1 assignment
    potential_merges = find_potential_merges(iou_matrix, gt_stones, iou_threshold)
    
    # Use Hungarian algorithm to find optimal 1:1 assignment
    # We want to MAXIMIZE IoU, but linear_sum_assignment MINIMIZES, so use negative
    cost_matrix = -iou_matrix
    
    # Handle case where n_gt != n_pred by padding
    n_gt, n_pred = iou_matrix.shape
    if n_gt > n_pred:
        # More GT than predictions - pad predictions with zeros
        padding = np.zeros((n_gt, n_gt - n_pred))
        cost_matrix = np.hstack([cost_matrix, padding])
    elif n_pred > n_gt:
        # More predictions than GT - pad GT with zeros
        padding = np.zeros((n_pred - n_gt, n_pred))
        cost_matrix = np.vstack([cost_matrix, padding])
    
    # Solve assignment
    gt_indices, pred_indices = linear_sum_assignment(cost_matrix)
    
    # Build assignment dict (only for valid indices)
    assignment = {}  # gt_idx -> pred_idx
    for gt_idx, pred_idx in zip(gt_indices, pred_indices):
        if gt_idx < len(gt_stones) and pred_idx < len(pred_stones):
            if iou_matrix[gt_idx, pred_idx] > 0:  # Only if there's actual overlap
                assignment[gt_idx] = pred_idx
    
    # Track which GT stones were "victims" of merging
    # A merge victim is a GT stone that overlaps with a predicted region
    # but lost the 1:1 assignment to another GT stone
    merge_victims = set()
    merge_groups = []  # List of (pred_idx, [gt_indices that wanted it])
    
    for pred_idx, gt_candidates in potential_merges.items():
        # Find which GT stone "won" this prediction
        winner = None
        for gt_idx in gt_candidates:
            if assignment.get(gt_idx) == pred_idx:
                winner = gt_idx
                break
        
        if winner is not None:
            # Others are merge victims
            victims = [gt_idx for gt_idx in gt_candidates if gt_idx != winner]
            # Only count as victims if they had decent overlap
            victims = [v for v in victims if iou_matrix[v, pred_idx] >= 0.1]
            if victims:
                merge_victims.update(victims)
                merge_groups.append({
                    'pred_stone_id': pred_stones[pred_idx].stone_id,
                    'winner_gt_id': gt_stones[winner].stone_id,
                    'merged_gt_ids': [gt_stones[v].stone_id for v in victims],
                    'all_gt_ids': [gt_stones[g].stone_id for g in gt_candidates]
                })
    
    # Build match results
    matches = []
    
    for gt_idx, gt_stone in enumerate(gt_stones):
        if gt_idx in assignment:
            pred_idx = assignment[gt_idx]
            pred_stone = pred_stones[pred_idx]
            iou = iou_matrix[gt_idx, pred_idx]
            
            is_detected = iou >= iou_threshold
            is_correct = is_detected and (pred_stone.class_id == gt_stone.class_id)
            
            # Find if this GT stone was the winner of a merge situation
            merged_with = []
            for mg in merge_groups:
                if mg['winner_gt_id'] == gt_stone.stone_id:
                    merged_with = mg['merged_gt_ids']
                    break
            
            match = StoneMatch(
                gt_stone=gt_stone,
                status='detected' if is_detected else 'undetected',
                iou=iou,
                predicted_class=pred_stone.class_id,
                predicted_stone_id=pred_stone.stone_id,
                is_correctly_classified=is_correct,
                merged_with_gt_ids=merged_with
            )
        elif gt_idx in merge_victims:
            # This stone was merged into another prediction
            # Find which prediction it overlapped with most
            best_pred_idx = np.argmax(iou_matrix[gt_idx, :])
            best_iou = iou_matrix[gt_idx, best_pred_idx]
            
            match = StoneMatch(
                gt_stone=gt_stone,
                status='merged',
                iou=best_iou,
                predicted_class=pred_stones[best_pred_idx].class_id if best_iou > 0 else None,
                predicted_stone_id=pred_stones[best_pred_idx].stone_id if best_iou > 0 else None,
                is_correctly_classified=False,  # Merged stones are not correctly detected
                merged_with_gt_ids=[]
            )
        else:
            # No overlap with any prediction
            match = StoneMatch(
                gt_stone=gt_stone,
                status='undetected',
                iou=0.0,
                predicted_class=None,
                predicted_stone_id=None,
                is_correctly_classified=False,
                merged_with_gt_ids=[]
            )
        
        matches.append(match)
    
    # Compile merge statistics
    merge_stats = {
        'total_merge_events': len(merge_groups),
        'merged_gt_stones': len(merge_victims),
        'merge_groups': merge_groups
    }
    
    return matches, merge_stats


print("Matching functions defined.")

---
## Cell 5: Metrics Calculation

In [ ]:
def calculate_detection_metrics(
    matches: List[StoneMatch],
    merge_stats: Dict,
    class_names: List[str]
) -> Dict:
    """
    Calculate stone-level detection metrics from matches.
    
    Returns dict with:
    - total_stones: Number of GT stones
    - detected_stones: Number detected (IoU >= threshold, 1:1 matched)
    - merged_stones: Number of GT stones lost to merging
    - undetected_stones: Number with no prediction overlap
    - detection_rate: detected / total
    - merge_rate: merged / total
    - correctly_classified: Number with correct class
    - classification_accuracy: correctly_classified / detected
    - per_class: Dict with per-class breakdown
    - mean_iou_detected: Mean IoU of detected stones
    """
    total = len(matches)
    detected = sum(1 for m in matches if m.status == 'detected')
    merged = sum(1 for m in matches if m.status == 'merged')
    undetected = sum(1 for m in matches if m.status == 'undetected')
    correctly_classified = sum(1 for m in matches if m.is_correctly_classified)
    
    # Per-class breakdown (only stone classes, not background)
    per_class = {}
    for class_idx, class_name in enumerate(class_names):
        if class_idx == 0:  # Skip background
            continue
        class_matches = [m for m in matches if m.gt_stone.class_id == class_idx]
        class_total = len(class_matches)
        class_detected = sum(1 for m in class_matches if m.status == 'detected')
        class_merged = sum(1 for m in class_matches if m.status == 'merged')
        class_correct = sum(1 for m in class_matches if m.is_correctly_classified)
        
        per_class[class_name] = {
            'total': class_total,
            'detected': class_detected,
            'merged': class_merged,
            'undetected': class_total - class_detected - class_merged,
            'detection_rate': class_detected / class_total if class_total > 0 else 0.0,
            'merge_rate': class_merged / class_total if class_total > 0 else 0.0,
            'correctly_classified': class_correct,
            'classification_accuracy': class_correct / class_detected if class_detected > 0 else 0.0
        }
    
    # Mean IoU of detected stones
    detected_ious = [m.iou for m in matches if m.status == 'detected']
    mean_iou_detected = np.mean(detected_ious) if detected_ious else 0.0
    
    return {
        'total_stones': total,
        'detected_stones': detected,
        'merged_stones': merged,
        'undetected_stones': undetected,
        'detection_rate': detected / total if total > 0 else 0.0,
        'merge_rate': merged / total if total > 0 else 0.0,
        'correctly_classified': correctly_classified,
        'classification_accuracy': correctly_classified / detected if detected > 0 else 0.0,
        'overall_accuracy': correctly_classified / total if total > 0 else 0.0,
        'per_class': per_class,
        'mean_iou_detected': mean_iou_detected,
        'merge_stats': merge_stats
    }


print("Metrics calculation defined.")

---
## Cell 6: Load Ground Truth and Extract Stones

In [ ]:
# Load ground truth
print("Loading ground truth mask...")
gt_rgb = np.array(Image.open(GT_MASK_PATH).convert('RGB'))
gt_mask = rgb_to_class_mask(gt_rgb)
print(f"  Shape: {gt_mask.shape}")
print(f"  Classes present: {np.unique(gt_mask)}")

# Extract individual stones
print(f"\nExtracting individual stones (min size = {MIN_STONE_SIZE} px)...")
gt_stones = extract_stones_from_mask(gt_mask, CLASS_NAMES, MIN_STONE_SIZE)
print(f"  Total stones found: {len(gt_stones)}")

# Per-class stone counts
print("\n  Per-class stone counts:")
for class_idx, class_name in enumerate(CLASS_NAMES):
    if class_idx == 0:  # Skip background
        continue
    count = sum(1 for s in gt_stones if s.class_id == class_idx)
    print(f"    {class_name}: {count}")

# Stone size statistics
sizes = [s.pixel_count for s in gt_stones]
print(f"\n  Stone size statistics (pixels):")
print(f"    Min: {min(sizes):,}")
print(f"    Max: {max(sizes):,}")
print(f"    Mean: {np.mean(sizes):,.0f}")
print(f"    Median: {np.median(sizes):,.0f}")

print("\n✓ Ground truth stones extracted.")

---
## Cell 7: Visualize Extracted GT Stones

In [ ]:
# Create visualization showing individual stones
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Original GT mask
cmap = ListedColormap(CLASS_COLORS)
axes[0].imshow(gt_mask, cmap=cmap, vmin=0, vmax=3)
axes[0].set_title(f'Ground Truth Mask\n({np.sum(gt_mask > 0):,} stone pixels)', fontsize=12, fontweight='bold')
axes[0].axis('off')

# Individual stone labels (random colors)
stone_label_img = np.zeros(gt_mask.shape, dtype=np.int32)
for stone in gt_stones:
    stone_label_img[stone.mask] = stone.stone_id

# Use a colormap that shows each stone distinctly
np.random.seed(42)
n_stones = len(gt_stones)
random_colors = np.random.rand(max(stone_label_img.max() + 1, n_stones + 1), 3)
random_colors[0] = [0, 0, 0]  # Background black
stone_cmap = ListedColormap(random_colors)

axes[1].imshow(stone_label_img, cmap=stone_cmap)
axes[1].set_title(f'Individual Stones (Instance Map)\n({len(gt_stones)} stones)', fontsize=12, fontweight='bold')
axes[1].axis('off')

# Stone size distribution histogram
sizes = [s.pixel_count for s in gt_stones]
axes[2].hist(sizes, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[2].axvline(np.mean(sizes), color='red', linestyle='--', label=f'Mean: {np.mean(sizes):,.0f}')
axes[2].axvline(np.median(sizes), color='orange', linestyle='--', label=f'Median: {np.median(sizes):,.0f}')
axes[2].set_xlabel('Stone Size (pixels)', fontsize=11)
axes[2].set_ylabel('Count', fontsize=11)
axes[2].set_title('Stone Size Distribution', fontsize=12, fontweight='bold')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## Cell 8: Load Predictions and Extract Predicted Stones

In [ ]:
# Load all prediction masks and extract stones
pred_masks = {}
pred_stones_dict = {}

print("Loading prediction masks and extracting stones...")
for model_name, pred_path in PRED_PATHS.items():
    print(f"\n  {model_name}:")
    pred_rgb = np.array(Image.open(pred_path).convert('RGB'))
    pred_mask = rgb_to_class_mask(pred_rgb, verbose=True)
    
    # Verify shape matches
    assert pred_mask.shape == gt_mask.shape, f"Shape mismatch for {model_name}: {pred_mask.shape} vs GT {gt_mask.shape}"
    
    pred_masks[model_name] = pred_mask
    
    # Extract predicted stones
    pred_stones = extract_stones_from_mask(pred_mask, CLASS_NAMES, min_size=MIN_STONE_SIZE)
    pred_stones_dict[model_name] = pred_stones
    
    print(f"    Shape: {pred_mask.shape}")
    print(f"    Predicted stones: {len(pred_stones)}")
    print(f"    GT stones: {len(gt_stones)}")
    print(f"    Difference: {len(pred_stones) - len(gt_stones):+d}")

print("\n✓ All predictions loaded and stones extracted.")

---
## Cell 9: Match Stones with 1:1 Constraint

In [ ]:
# Match stones and calculate metrics for each model
all_results = {}
all_matches = {}
all_merge_stats = {}

print(f"Matching stones with 1:1 constraint (IoU threshold = {IOU_THRESHOLD})...\n")
print("="*80)

for model_name in PRED_PATHS.keys():
    print(f"\n{model_name.upper()}")
    print("-"*50)
    
    pred_mask = pred_masks[model_name]
    pred_stones = pred_stones_dict[model_name]
    
    # Match GT stones to predictions with 1:1 constraint
    matches, merge_stats = match_stones_one_to_one(
        gt_stones, pred_stones, pred_mask, IOU_THRESHOLD
    )
    all_matches[model_name] = matches
    all_merge_stats[model_name] = merge_stats
    
    # Calculate metrics
    metrics = calculate_detection_metrics(matches, merge_stats, CLASS_NAMES)
    all_results[model_name] = metrics
    
    # Print results
    print(f"  Total GT stones: {metrics['total_stones']}")
    print(f"  Predicted stones: {len(pred_stones)}")
    print(f"")
    print(f"  Detected (1:1 matched): {metrics['detected_stones']} ({metrics['detection_rate']*100:.1f}%)")
    print(f"  Merged (lost to merge): {metrics['merged_stones']} ({metrics['merge_rate']*100:.1f}%)")
    print(f"  Undetected (no overlap): {metrics['undetected_stones']}")
    print(f"")
    print(f"  Correctly classified: {metrics['correctly_classified']} ({metrics['overall_accuracy']*100:.1f}% of total)")
    print(f"  Classification accuracy (of detected): {metrics['classification_accuracy']*100:.1f}%")
    print(f"  Mean IoU (detected): {metrics['mean_iou_detected']:.4f}")
    
    # Merge details
    if merge_stats['total_merge_events'] > 0:
        print(f"")
        print(f"  ⚠ MERGE EVENTS: {merge_stats['total_merge_events']}")
        print(f"    GT stones affected by merging: {merge_stats['merged_gt_stones']}")
    
    print("\n  Per-class breakdown:")
    for class_name, class_metrics in metrics['per_class'].items():
        if class_metrics['total'] > 0:
            print(f"    {class_name:15s}: {class_metrics['detected']:3d}/{class_metrics['total']:3d} detected "
                  f"({class_metrics['detection_rate']*100:5.1f}%), "
                  f"{class_metrics['merged']:2d} merged, "
                  f"{class_metrics['correctly_classified']:3d} correct")

print("\n" + "="*80)
print("✓ All stone matching complete.")

---
## Cell 10: Comparative Summary Table

In [ ]:
# Build comparative summary table
summary_data = []

for model_name in PRED_PATHS.keys():
    metrics = all_results[model_name]
    row = {
        'Model': model_name,
        'GT_Stones': metrics['total_stones'],
        'Pred_Stones': len(pred_stones_dict[model_name]),
        'Detected': metrics['detected_stones'],
        'Merged': metrics['merged_stones'],
        'Undetected': metrics['undetected_stones'],
        'Detection_Rate': metrics['detection_rate'],
        'Merge_Rate': metrics['merge_rate'],
        'Correct_Class': metrics['correctly_classified'],
        'Overall_Accuracy': metrics['overall_accuracy'],
        'Mean_IoU': metrics['mean_iou_detected']
    }
    
    # Add per-class detection rates
    for class_name in ['Ashlar', 'Polygonal', 'Quarry Stone']:
        if class_name in metrics['per_class']:
            row[f'{class_name}_Det'] = metrics['per_class'][class_name]['detection_rate']
            row[f'{class_name}_Mrg'] = metrics['per_class'][class_name]['merge_rate']
        else:
            row[f'{class_name}_Det'] = np.nan
            row[f'{class_name}_Mrg'] = np.nan
    
    summary_data.append(row)

summary_df = pd.DataFrame(summary_data)

# Display
print("\n" + "="*120)
print(f"STONE-LEVEL DETECTION SUMMARY (IoU Threshold = {IOU_THRESHOLD}, 1:1 Matching)")
print("="*120)

# Format for display
display_cols = ['Model', 'GT_Stones', 'Pred_Stones', 'Detected', 'Merged', 'Undetected', 
                'Detection_Rate', 'Merge_Rate', 'Overall_Accuracy', 'Mean_IoU']
display_df = summary_df[display_cols].copy()

pct_cols = ['Detection_Rate', 'Merge_Rate', 'Overall_Accuracy']
for col in pct_cols:
    display_df[col] = display_df[col].apply(lambda x: f'{x*100:.1f}%')
display_df['Mean_IoU'] = display_df['Mean_IoU'].apply(lambda x: f'{x:.3f}')

print(display_df.to_string(index=False))
print("="*120)

# Highlight best model
best_det_model = summary_df.loc[summary_df['Detection_Rate'].idxmax(), 'Model']
best_det_rate = summary_df['Detection_Rate'].max()
lowest_merge_model = summary_df.loc[summary_df['Merge_Rate'].idxmin(), 'Model']
lowest_merge_rate = summary_df['Merge_Rate'].min()

print(f"\n→ Best detection rate: {best_det_model} ({best_det_rate*100:.1f}%)")
print(f"→ Lowest merge rate: {lowest_merge_model} ({lowest_merge_rate*100:.1f}%)")

---
## Cell 11: Detection & Merge Rate Bar Charts

In [ ]:
# Stacked bar chart showing detected / merged / undetected breakdown
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

model_names = list(PRED_PATHS.keys())
x = np.arange(len(model_names))
width = 0.6

# Left plot: Stacked bar (detected + merged + undetected)
detected_counts = [all_results[m]['detected_stones'] for m in model_names]
merged_counts = [all_results[m]['merged_stones'] for m in model_names]
undetected_counts = [all_results[m]['undetected_stones'] for m in model_names]

bars1 = axes[0].bar(x, detected_counts, width, label='Detected', color='#2ecc71', edgecolor='black')
bars2 = axes[0].bar(x, merged_counts, width, bottom=detected_counts, label='Merged', color='#f39c12', edgecolor='black')
bars3 = axes[0].bar(x, undetected_counts, width, 
                    bottom=[d+m for d,m in zip(detected_counts, merged_counts)], 
                    label='Undetected', color='#e74c3c', edgecolor='black')

# Add count labels
for i, (d, m, u) in enumerate(zip(detected_counts, merged_counts, undetected_counts)):
    total = d + m + u
    axes[0].text(i, d/2, f'{d}', ha='center', va='center', fontweight='bold', fontsize=11, color='white')
    if m > 0:
        axes[0].text(i, d + m/2, f'{m}', ha='center', va='center', fontweight='bold', fontsize=11)
    if u > 0:
        axes[0].text(i, d + m + u/2, f'{u}', ha='center', va='center', fontweight='bold', fontsize=11, color='white')

axes[0].set_ylabel('Number of GT Stones', fontsize=12)
axes[0].set_title('Stone Detection Breakdown\n(1:1 Matching)', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(model_names, fontsize=11)
axes[0].legend(loc='upper right', fontsize=10)
axes[0].grid(axis='y', alpha=0.3)

# Right plot: Detection rate vs Merge rate
detection_rates = [all_results[m]['detection_rate'] * 100 for m in model_names]
merge_rates = [all_results[m]['merge_rate'] * 100 for m in model_names]

x2 = np.arange(len(model_names))
width2 = 0.35

bars_det = axes[1].bar(x2 - width2/2, detection_rates, width2, label='Detection Rate', 
                       color='#2ecc71', edgecolor='black', linewidth=1.5)
bars_mrg = axes[1].bar(x2 + width2/2, merge_rates, width2, label='Merge Rate', 
                       color='#f39c12', edgecolor='black', linewidth=1.5)

for bar, val in zip(bars_det, detection_rates):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                 f'{val:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=10)
for bar, val in zip(bars_mrg, merge_rates):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                 f'{val:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=10)

axes[1].set_ylabel('Rate (%)', fontsize=12)
axes[1].set_title(f'Detection vs Merge Rate\n(IoU ≥ {IOU_THRESHOLD})', fontsize=14, fontweight='bold')
axes[1].set_xticks(x2)
axes[1].set_xticklabels(model_names, fontsize=11)
axes[1].legend(loc='upper right', fontsize=10)
axes[1].set_ylim(0, 110)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---
## Cell 12: Per-Class Detection with Merge Breakdown

In [ ]:
# Grouped bar chart for per-class detection and merge rates
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

stone_classes = ['Ashlar', 'Polygonal', 'Quarry Stone']
x = np.arange(len(stone_classes))
width = 0.25

# Detection rates
for idx, model_name in enumerate(PRED_PATHS.keys()):
    detection_rates = []
    for class_name in stone_classes:
        if class_name in all_results[model_name]['per_class']:
            rate = all_results[model_name]['per_class'][class_name]['detection_rate'] * 100
        else:
            rate = 0
        detection_rates.append(rate)
    
    offset = width * idx
    bars = axes[0].bar(x + offset, detection_rates, width, 
                       label=model_name, color=MODEL_COLORS[model_name],
                       edgecolor='black', linewidth=1)
    
    for bar, val in zip(bars, detection_rates):
        axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                     f'{val:.0f}%', ha='center', va='bottom', fontsize=8, fontweight='bold')

axes[0].set_ylabel('Detection Rate (%)', fontsize=12)
axes[0].set_title(f'Per-Class Detection Rate', fontsize=14, fontweight='bold')
axes[0].set_xticks(x + width)
axes[0].set_xticklabels(stone_classes, fontsize=11)
axes[0].legend(loc='upper right', fontsize=10)
axes[0].set_ylim(0, 115)
axes[0].grid(axis='y', alpha=0.3)

# Merge rates
for idx, model_name in enumerate(PRED_PATHS.keys()):
    merge_rates = []
    for class_name in stone_classes:
        if class_name in all_results[model_name]['per_class']:
            rate = all_results[model_name]['per_class'][class_name]['merge_rate'] * 100
        else:
            rate = 0
        merge_rates.append(rate)
    
    offset = width * idx
    bars = axes[1].bar(x + offset, merge_rates, width, 
                       label=model_name, color=MODEL_COLORS[model_name],
                       edgecolor='black', linewidth=1)
    
    for bar, val in zip(bars, merge_rates):
        if val > 0:
            axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                         f'{val:.0f}%', ha='center', va='bottom', fontsize=8, fontweight='bold')

axes[1].set_ylabel('Merge Rate (%)', fontsize=12)
axes[1].set_title(f'Per-Class Merge Rate (lower is better)', fontsize=14, fontweight='bold')
axes[1].set_xticks(x + width)
axes[1].set_xticklabels(stone_classes, fontsize=11)
axes[1].legend(loc='upper right', fontsize=10)
axes[1].grid(axis='y', alpha=0.3)

# Add stone counts
for i, class_name in enumerate(stone_classes):
    first_model = list(PRED_PATHS.keys())[0]
    if class_name in all_results[first_model]['per_class']:
        total = all_results[first_model]['per_class'][class_name]['total']
        axes[0].text(i + width, -8, f'(n={total})', ha='center', fontsize=10, color='gray')
        axes[1].text(i + width, -3, f'(n={total})', ha='center', fontsize=10, color='gray')

plt.tight_layout()
plt.show()

---
## Cell 13: Detection Result Visualization (with Merge Highlighting)

In [ ]:
# Create detection visualization for each model
# Green = correctly detected and classified
# Yellow/Orange = detected but wrong class OR merged
# Red = not detected at all

fig, axes = plt.subplots(2, 2, figsize=(16, 16))

# Ground truth
cmap = ListedColormap(CLASS_COLORS)
axes[0, 0].imshow(gt_mask, cmap=cmap, vmin=0, vmax=3)
axes[0, 0].set_title(f'Ground Truth\n({len(gt_stones)} stones)', fontsize=12, fontweight='bold')
axes[0, 0].axis('off')

# Detection results for each model
model_list = list(PRED_PATHS.keys())
positions = [(0, 1), (1, 0), (1, 1)]

for (row, col), model_name in zip(positions, model_list):
    matches = all_matches[model_name]
    metrics = all_results[model_name]
    
    # Create RGB visualization
    vis_img = np.zeros((*gt_mask.shape, 3), dtype=np.uint8)
    vis_img[gt_mask == 0] = [30, 30, 30]  # Dark gray background
    
    for match in matches:
        if match.is_correctly_classified:
            color = [0, 200, 0]      # Green - correctly detected and classified
        elif match.status == 'detected':
            color = [255, 200, 0]    # Yellow - detected but wrong class
        elif match.status == 'merged':
            color = [255, 128, 0]    # Orange - merged with another stone
        else:
            color = [200, 0, 0]      # Red - not detected
        
        vis_img[match.gt_stone.mask] = color
    
    axes[row, col].imshow(vis_img)
    axes[row, col].set_title(
        f'{model_name}\n'
        f'Detected: {metrics["detection_rate"]*100:.1f}% | '
        f'Merged: {metrics["merge_rate"]*100:.1f}% | '
        f'Overall Acc: {metrics["overall_accuracy"]*100:.1f}%',
        fontsize=11, fontweight='bold'
    )
    axes[row, col].axis('off')

# Add legend
legend_elements = [
    Patch(facecolor='#00C800', label='Correct (detected + right class)'),
    Patch(facecolor='#FFC800', label='Detected but wrong class'),
    Patch(facecolor='#FF8000', label='Merged (lost to another stone)'),
    Patch(facecolor='#C80000', label='Undetected (no overlap)')
]
fig.legend(handles=legend_elements, loc='lower center', ncol=4, fontsize=10, 
           bbox_to_anchor=(0.5, 0.02))

plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()

---
## Cell 14: Merge Event Details

In [ ]:
# Show details of merge events for each model
print("="*80)
print("MERGE EVENT DETAILS")
print("="*80)

for model_name in PRED_PATHS.keys():
    merge_stats = all_merge_stats[model_name]
    
    print(f"\n{model_name.upper()}")
    print("-"*50)
    
    if merge_stats['total_merge_events'] == 0:
        print("  No merge events detected. ✓")
    else:
        print(f"  Total merge events: {merge_stats['total_merge_events']}")
        print(f"  GT stones lost to merging: {merge_stats['merged_gt_stones']}")
        print("")
        
        for i, mg in enumerate(merge_stats['merge_groups'], 1):
            print(f"  Merge #{i}:")
            print(f"    Predicted stone ID: {mg['pred_stone_id']}")
            print(f"    Winner GT stone ID: {mg['winner_gt_id']}")
            print(f"    Merged GT stone IDs: {mg['merged_gt_ids']}")
            print(f"    Total GT stones involved: {len(mg['all_gt_ids'])}")

print("\n" + "="*80)

---
## Cell 15: Visualize Merge Events

In [ ]:
# Visualize merged stones specifically
# Show which GT stones got merged together

# Find the model with the most merges for demonstration
merge_counts = {m: all_merge_stats[m]['merged_gt_stones'] for m in PRED_PATHS.keys()}
max_merge_model = max(merge_counts, key=merge_counts.get)
min_merge_model = min(merge_counts, key=merge_counts.get)

if merge_counts[max_merge_model] > 0:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # GT
    axes[0].imshow(gt_mask, cmap=ListedColormap(CLASS_COLORS), vmin=0, vmax=3)
    axes[0].set_title(f'Ground Truth', fontsize=12, fontweight='bold')
    axes[0].axis('off')
    
    # Model with most merges - highlight merged stones
    matches = all_matches[max_merge_model]
    merge_stats = all_merge_stats[max_merge_model]
    
    vis_img = np.zeros((*gt_mask.shape, 3), dtype=np.uint8)
    vis_img[gt_mask == 0] = [30, 30, 30]
    
    # Color merged stones by their merge group
    merge_colors = plt.cm.tab10(np.linspace(0, 1, max(len(merge_stats['merge_groups']), 1) + 1))
    
    # First, color all detected stones gray
    for match in matches:
        if match.status == 'detected':
            vis_img[match.gt_stone.mask] = [100, 100, 100]
    
    # Then highlight merge groups with distinct colors
    for i, mg in enumerate(merge_stats['merge_groups']):
        color = (np.array(merge_colors[i][:3]) * 255).astype(np.uint8)
        
        # Color all GT stones in this merge group
        for gt_id in mg['all_gt_ids']:
            for match in matches:
                if match.gt_stone.stone_id == gt_id:
                    vis_img[match.gt_stone.mask] = color
                    break
    
    axes[1].imshow(vis_img)
    axes[1].set_title(f'{max_merge_model} - Merge Groups Highlighted\n'
                      f'({merge_stats["total_merge_events"]} merge events, '
                      f'{merge_stats["merged_gt_stones"]} GT stones affected)',
                      fontsize=12, fontweight='bold')
    axes[1].axis('off')
    
    # Model with least merges for comparison
    matches_min = all_matches[min_merge_model]
    merge_stats_min = all_merge_stats[min_merge_model]
    
    vis_img2 = np.zeros((*gt_mask.shape, 3), dtype=np.uint8)
    vis_img2[gt_mask == 0] = [30, 30, 30]
    
    for match in matches_min:
        if match.status == 'detected':
            vis_img2[match.gt_stone.mask] = [100, 100, 100]
    
    for i, mg in enumerate(merge_stats_min['merge_groups']):
        color = (np.array(merge_colors[i][:3]) * 255).astype(np.uint8)
        for gt_id in mg['all_gt_ids']:
            for match in matches_min:
                if match.gt_stone.stone_id == gt_id:
                    vis_img2[match.gt_stone.mask] = color
                    break
    
    axes[2].imshow(vis_img2)
    axes[2].set_title(f'{min_merge_model} - Merge Groups Highlighted\n'
                      f'({merge_stats_min["total_merge_events"]} merge events, '
                      f'{merge_stats_min["merged_gt_stones"]} GT stones affected)',
                      fontsize=12, fontweight='bold')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print("No merge events to visualize - all models correctly separated stones!")

---
## Cell 16: Threshold Sensitivity Analysis

In [ ]:
# Analyze how detection rate changes with different IoU thresholds
# Note: Merge detection is done at threshold 0.1 overlap, so merge counts stay constant

thresholds = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for model_name in PRED_PATHS.keys():
    matches = all_matches[model_name]
    
    detection_rates = []
    overall_accs = []
    
    for thresh in thresholds:
        # Count detected at this threshold (excluding merged)
        detected = sum(1 for m in matches if m.status != 'merged' and m.iou >= thresh)
        correct = sum(1 for m in matches if m.status != 'merged' and m.iou >= thresh and 
                      m.predicted_class == m.gt_stone.class_id)
        
        total_valid = len(matches) - sum(1 for m in matches if m.status == 'merged')
        
        detection_rates.append(detected / len(matches) * 100)
        overall_accs.append(correct / len(matches) * 100)
    
    axes[0].plot(thresholds, detection_rates, 'o-', color=MODEL_COLORS[model_name], 
                 label=model_name, linewidth=2, markersize=8)
    axes[1].plot(thresholds, overall_accs, 'o-', color=MODEL_COLORS[model_name],
                 label=model_name, linewidth=2, markersize=8)

# Mark current threshold
for ax in axes:
    ax.axvline(IOU_THRESHOLD, color='gray', linestyle='--', alpha=0.7, 
               label=f'Current threshold ({IOU_THRESHOLD})')

axes[0].set_xlabel('IoU Threshold', fontsize=12)
axes[0].set_ylabel('Detection Rate (%)', fontsize=12)
axes[0].set_title('Detection Rate vs IoU Threshold', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)
axes[0].set_ylim(0, 105)

axes[1].set_xlabel('IoU Threshold', fontsize=12)
axes[1].set_ylabel('Overall Accuracy (%)', fontsize=12)
axes[1].set_title('Overall Accuracy vs IoU Threshold', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)
axes[1].set_ylim(0, 105)

plt.tight_layout()
plt.show()

---
## Cell 17: Save Results

In [ ]:
# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save summary table
summary_path = os.path.join(OUTPUT_DIR, 'stone_detection_summary.csv')
summary_df.to_csv(summary_path, index=False)
print(f"✓ Summary table saved to: {summary_path}")

# Save detailed per-stone results for each model
for model_name in PRED_PATHS.keys():
    matches = all_matches[model_name]
    
    stone_data = []
    for match in matches:
        stone_data.append({
            'stone_id': match.gt_stone.stone_id,
            'gt_class': match.gt_stone.class_name,
            'gt_class_id': match.gt_stone.class_id,
            'pixel_count': match.gt_stone.pixel_count,
            'centroid_row': match.gt_stone.centroid[0],
            'centroid_col': match.gt_stone.centroid[1],
            'status': match.status,
            'iou': match.iou,
            'predicted_stone_id': match.predicted_stone_id,
            'predicted_class': CLASS_NAMES[match.predicted_class] if match.predicted_class is not None else 'None',
            'predicted_class_id': match.predicted_class if match.predicted_class is not None else -1,
            'is_correctly_classified': match.is_correctly_classified,
            'merged_with_gt_ids': str(match.merged_with_gt_ids) if match.merged_with_gt_ids else ''
        })
    
    stone_df = pd.DataFrame(stone_data)
    stone_path = os.path.join(OUTPUT_DIR, f'stone_detection_{model_name.replace("-", "_")}.csv')
    stone_df.to_csv(stone_path, index=False)
    print(f"✓ Per-stone results for {model_name} saved to: {stone_path}")

# Save merge event details
merge_data = []
for model_name in PRED_PATHS.keys():
    merge_stats = all_merge_stats[model_name]
    for mg in merge_stats['merge_groups']:
        merge_data.append({
            'model': model_name,
            'pred_stone_id': mg['pred_stone_id'],
            'winner_gt_id': mg['winner_gt_id'],
            'merged_gt_ids': str(mg['merged_gt_ids']),
            'num_merged': len(mg['merged_gt_ids']),
            'total_gt_involved': len(mg['all_gt_ids'])
        })

if merge_data:
    merge_df = pd.DataFrame(merge_data)
    merge_path = os.path.join(OUTPUT_DIR, 'merge_events.csv')
    merge_df.to_csv(merge_path, index=False)
    print(f"✓ Merge events saved to: {merge_path}")
else:
    print("  No merge events to save.")

# Save threshold sensitivity data
threshold_data = []
for thresh in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    row = {'threshold': thresh}
    for model_name in PRED_PATHS.keys():
        matches = all_matches[model_name]
        detected = sum(1 for m in matches if m.status != 'merged' and m.iou >= thresh)
        correct = sum(1 for m in matches if m.status != 'merged' and m.iou >= thresh and 
                      m.predicted_class == m.gt_stone.class_id)
        merged = sum(1 for m in matches if m.status == 'merged')
        
        row[f'{model_name}_detection_rate'] = detected / len(matches)
        row[f'{model_name}_overall_acc'] = correct / len(matches)
        row[f'{model_name}_merge_rate'] = merged / len(matches)
    threshold_data.append(row)

threshold_df = pd.DataFrame(threshold_data)
threshold_path = os.path.join(OUTPUT_DIR, 'threshold_sensitivity.csv')
threshold_df.to_csv(threshold_path, index=False)
print(f"✓ Threshold sensitivity data saved to: {threshold_path}")

print("\n✓ All results saved.")

---
## Cell 18: Summary for Paper

In [ ]:
# Generate summary text suitable for paper
print("="*80)
print("SUMMARY FOR PAPER")
print("="*80)

print(f"\nDataset: {len(gt_stones)} individual stones across {len([c for c in np.unique(gt_mask) if c > 0])} masonry classes")
print(f"Evaluation: Stone-level detection with 1:1 matching constraint")
print(f"IoU Threshold: {IOU_THRESHOLD}")
print(f"Minimum stone size: {MIN_STONE_SIZE} pixels\n")

print("Per-class stone counts:")
for class_name in ['Ashlar', 'Polygonal', 'Quarry Stone']:
    count = sum(1 for s in gt_stones if s.class_name == class_name)
    print(f"  {class_name}: {count}")

print("\n" + "-"*80)
print("Model Comparison (Stone-Level Metrics with 1:1 Matching):")
print("-"*80)

for model_name in PRED_PATHS.keys():
    metrics = all_results[model_name]
    print(f"\n{model_name}:")
    print(f"  Predicted stones: {len(pred_stones_dict[model_name])} (GT: {metrics['total_stones']})")
    print(f"  Stone Detection Rate: {metrics['detection_rate']*100:.1f}%")
    print(f"  Stone Merge Rate: {metrics['merge_rate']*100:.1f}%")
    print(f"  Classification Accuracy (of detected): {metrics['classification_accuracy']*100:.1f}%")
    print(f"  Overall Accuracy (detected + correct): {metrics['overall_accuracy']*100:.1f}%")
    print(f"  Mean IoU of detected stones: {metrics['mean_iou_detected']:.3f}")

print("\n" + "-"*80)
print("Key Insight:")
print("-"*80)

# Find most notable difference
merge_rates = {m: all_results[m]['merge_rate'] for m in PRED_PATHS.keys()}
if max(merge_rates.values()) - min(merge_rates.values()) > 0.05:
    worst_merge = max(merge_rates, key=merge_rates.get)
    best_merge = min(merge_rates, key=merge_rates.get)
    print(f"\nThe {best_merge} model shows the lowest merge rate ({merge_rates[best_merge]*100:.1f}%),")
    print(f"indicating better stone separation compared to {worst_merge} ({merge_rates[worst_merge]*100:.1f}%).")
    print(f"\nThis suggests that {'geometric information helps' if 'channel' in best_merge and '3' in best_merge else 'the combined approach helps'}")
    print(f"the model learn to distinguish individual stones rather than merging adjacent blocks.")
else:
    print("\nAll models show similar merge rates, indicating comparable stone separation ability.")

print("\n" + "="*80)